# 02D — Does exhaustive training improve LOSO Musanga detection and cover?

**Matched test sites:** Baego, MOS (1–2 and 5–6), Yangambi, Yoko. MOS 3–4 excluded completely. M2 uses **partial tree labels plus exhaustive circles of TRAINING sites only**; entire test site is withheld. M1 from 02A is the paired comparison. Native RGB TinyUNet, four M2 fits and no illumination sweep. Outputs are separate. Run 02A and 02B first for matching baseline.


## 01 — Input registry (34 Nadine circles, six mosaics)

In [ ]:
%pip -q install rasterio geopandas pyogrio shapely pandas matplotlib scikit-learn
from pathlib import Path
import json, warnings, math, random, time
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import average_precision_score, roc_auc_score
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                         Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None
OUT=ROOT/'Outputs/V3/02D_M2_LOSO_no_MOS34';OUT.mkdir(parents=True,exist_ok=True)
REF=ROOT/'Outputs/V3/01_reference_data_no_MOS34'
A=json.loads((ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json').read_text())
assert A.get('schema_version')=='v3_input_audit_5_nadine_circles'
assert A['exhaustive_annotations'].get('all_annotation_polygons_musanga') is True
assert int(A['exhaustive_annotations']['verified_exhaustive_circle_count'])==34
EXCLUDED_MOSAIC='Mos_3_4_transparent_mosaic_group1_georef.tif'
mosaics={m['file']:m for m in A['mosaics'] if m['file']!=EXCLUDED_MOSAIC}
splits=pd.read_csv(REF/'01_cv_window_manifest.csv')
links=pd.read_csv(REF/'01_circle_spatial_folds.csv')
repairs=pd.read_csv(REF/'01_geometry_repairs.csv')
assert links.circle_id.nunique()==30
assert links.groupby('site').circle_id.nunique().to_dict()=={'Baego':7,'MOS':11,'Yangambi':7,'Yoko':5}
assert links.circle_id.is_unique
assert len(mosaics)==5,[(k,v['site']) for k,v in mosaics.items()]
FOLD=0;SEED=20261016;CROP=256;EPOCHS=25;BATCH=8
MAX_PARTIAL_PER_MOSAIC=250
MAX_EXHAUSTIVE_PER_SITE=260
HOLDOUT_COUNTS={'Baego':1,'MOS':2,'Yangambi':2,'Yoko':2}
assert torch.cuda.is_available(),'Select Runtime > Change runtime type > GPU for the pilot.'
device='cuda';torch.manual_seed(SEED);np.random.seed(SEED);random.seed(SEED)
print('M1 LOSO partial-only: five mosaics, four sites; 30 circles protected from training')
print('GPU',torch.cuda.get_device_name(0),'| fold',FOLD)
print('Train/val candidates:',splits[splits.cv_fold==FOLD].split.value_counts().to_dict())
print('Files:',ROOT,'| output:',OUT)




print('Mosaics:',[(name,m['site']) for name,m in mosaics.items()])


## 02 — Load annotations and TinyUNet

In [ ]:
def pieces(g):
    if g.geom_type in ('Polygon','MultiPolygon'):return [g]
    if hasattr(g,'geoms'):return [p for q in g.geoms for p in pieces(q)]
    return []
def polygons(path):
    src=gpd.read_file(path);inds=[];shapes=[]
    for i,g in enumerate(src.geometry):
        if g is None or g.is_empty:continue
        if not g.is_valid:
            try:
                fixed=unary_union(pieces(make_valid(g)))
                change=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
                if change>(.01 if Path(path) in (Path(A['exhaustive_annotations']['complete_roi_boundaries']),Path(A['exhaustive_annotations']['positive_polygons']),Path(A['roi'])) else .05) or not fixed.is_valid:continue
                g=fixed
            except Exception:continue
        inds.append(i);shapes.append(g)
    x=src.iloc[inds].copy().reset_index(drop=True)
    x.geometry=shapes
    x['original_row']=inds
    return x
roi=polygons(A['roi'])
circles=polygons(A['exhaustive_annotations']['complete_roi_boundaries'])
positive=polygons(A['exhaustive_annotations']['positive_polygons'])
assert len(circles)==34 and len(positive)==1368,(
    'Unexpected reference geometry count; do not train using a partial dataset')
circle_ids=set(links.circle_id.astype(int))
circles=circles[circles.original_row.isin(circle_ids)].copy()
tree={}
for site,path in A['shared_annotations'].items():
    t=polygons(path)
    cls=pd.to_numeric(t.Class_ID,errors='coerce')
    t['class01']=np.where(cls.eq(1),1,np.where(cls.notna(),0,-1))
    tree[site]=t
geo={}
for name,m in mosaics.items():
    with rasterio.open(m['path']) as src:
        c=circles.to_crs(src.crs).copy()
        c['fold']=c.original_row.map(links.set_index('circle_id').fold)
        c['owner']=c.original_row.map(links.set_index('circle_id').mosaic)
        geo[name]=dict(site=m['site'],path=m['path'],transform=src.transform,
            crs=src.crs,roi=roi.to_crs(src.crs),
            circles=c,positive=positive.to_crs(src.crs),trees=tree[m['site']].to_crs(src.crs))



In [ ]:
class TinyUNet(nn.Module):
    def __init__(self,base=16):
        super().__init__()
        def block(a,b):
            return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True),
                nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
        self.e1=block(3,base);self.e2=block(base,base*2)
        self.bridge=block(base*2,base*4)
        self.d2=block(base*4+base*2,base*2)
        self.d1=block(base*2+base,base)
        self.final=nn.Conv2d(base,1,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(F.max_pool2d(e1,2))
        b=self.bridge(F.max_pool2d(e2,2))
        x=self.d2(torch.cat([F.interpolate(b,size=e2.shape[-2:],mode='bilinear',align_corners=False),e2],1))
        x=self.d1(torch.cat([F.interpolate(x,size=e1.shape[-2:],mode='bilinear',align_corners=False),e1],1))
        return self.final(x)[:,0]
class CropData(Dataset):
    def __init__(self,rows,augment=False):self.rows=rows;self.augment=augment
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        d=self.rows[i];image=d['rgb'];y=d['y'];w=d['w']
        if self.augment:
            rng=np.random.default_rng(SEED+i+int(torch.randint(0,1000000,(1,)).item()))
            k=int(rng.integers(4));image=np.rot90(image,k);y=np.rot90(y,k);w=np.rot90(w,k)
            if rng.random()<.5:image=image[:,::-1];y=y[:,::-1];w=w[:,::-1]
        x=torch.tensor(np.ascontiguousarray(image.transpose(2,0,1)),dtype=torch.float32)
        return x,torch.tensor(np.ascontiguousarray(y)),torch.tensor(np.ascontiguousarray(w))
def masked_loss(logits,y,w):
    loss=F.binary_cross_entropy_with_logits(logits,y,reduction='none')
    return (loss*w).sum()/w.sum().clamp_min(1.)
print('Tiny U-Net ready (native 256×256 RGB crops).')




## 03 — Strict training exclusion: all 34 exhaustive circles (including MOS cross-mosaic buffers)

In [ ]:
HOLDOUT=set(links.circle_id.astype(int))
DEV=set()
split_df=links[['circle_id','site','mosaic','cover_pct']].copy()
split_df['split']='holdout'
BLOCKED={}
for name,d in geo.items():
    this=circles.to_crs(d['crs'])
    # 15m exclusion across the complete site, not only the mosaic assigned to a circle.
    site_ids=set(links.loc[links.site==d['site'],'circle_id'].astype(int))
    BLOCKED[name]=[g.buffer(15.) for g in this[this.original_row.isin(site_ids)].geometry]
assert len(HOLDOUT)==30
print('All',len(HOLDOUT),'circles excluded from training, across MOS mosaic boundaries.')


## 04 — Fast preparation: local SSD staging by mosaic and restartable Drive chip caches

For each mosaic, the first run copies **only that GeoTIFF** to Colab local SSD when space permits, reuses a single open raster handle, saves prepared chips in batches of 100 windows to Drive, and deletes the staged copy. When space is insufficient, it uses one open Drive handle. Saved chip chunks can be reused after a disconnect.

The first run still requires raster reads. **Do not interrupt an already-progressing older NB02F session just to adopt this version.**


In [ ]:
from rasterio.windows import Window
from torch.utils.data import Dataset,DataLoader
candidate=pd.read_csv(REF/'01_candidate_windows.csv')
candidate=candidate[candidate.mosaic.isin(mosaics)].copy()
all_circle_buffers={}
for name,d in geo.items():
    cs=circles.to_crs(d['crs'])
    all_circle_buffers[name]=[g.buffer(15.) for g in cs.geometry]
    d['dev_circles']=cs[cs.original_row.isin(DEV)]
    d['holdout_circles']=cs[cs.original_row.isin(HOLDOUT)]
    d['positive_local']=positive.to_crs(d['crs'])
    d['trees_local']=tree[d['site']].to_crs(d['crs'])
    d['tree_positive']=d['trees_local'].loc[d['trees_local'].class01==1]
    d['tree_negative']=d['trees_local'].loc[d['trees_local'].class01==0]
    d['roi_local']=roi.to_crs(d['crs'])

def burn_window(geometries,name,win):
    """Rasterize geometries intersecting a crop, using the spatial index.

    Accept GeoSeries OR GeoDataFrame; always iterate actual Shapely geometries,
    never GeoDataFrame column names.
    """
    d=geo[name]
    series=geometries.geometry if isinstance(geometries,gpd.GeoDataFrame) else geometries
    assert isinstance(series,gpd.GeoSeries),type(series)
    bb=box(*rasterio.windows.bounds(win,d['transform']))
    ids=series.sindex.query(bb,predicate='intersects')
    shape=(int(win.height),int(win.width))
    if len(ids)==0:return np.zeros(shape,dtype=bool)
    selected=series.iloc[ids]
    return rasterize(((g,1) for g in selected.values
                      if g is not None and not g.is_empty),
        out_shape=shape,transform=rasterio.windows.transform(win,d['transform']),
        fill=0,dtype='uint8').astype(bool)

_ACTIVE_RASTER=None
_ACTIVE_NAME=None

def rgb_for_window(name,win):
    if _ACTIVE_RASTER is not None and _ACTIVE_NAME==name:
        raw=_ACTIVE_RASTER.read([1,2,3],window=win,masked=True)
    else:
        with rasterio.open(geo[name]['path']) as ds:
            raw=ds.read([1,2,3],window=win,masked=True)
    valid=~np.any(np.ma.getmaskarray(raw),axis=0)
    a=raw.filled(0).astype('float32')
    valid &= np.any(a!=0,axis=0)&np.any(a!=255,axis=0)
    valid &= ~((a[0]==0)&(a[1]==0)&(a[2]==255))
    return np.clip(a/255.,0,1).transpose(1,2,0).astype('float32'),valid

def training_crop(rec,model):
    name=rec.mosaic;d=geo[name]
    if rec.height<CROP or rec.width<CROP:return None
    rr=int(rec.row)+(int(rec.height)-CROP)//2
    cc=int(rec.col)+(int(rec.width)-CROP)//2
    win=Window(cc,rr,CROP,CROP)
    bb=box(*rasterio.windows.bounds(win,d['transform']))
    if any(bb.intersects(g) for g in BLOCKED[name]):return None
    if model=='M1' and any(bb.intersects(g) for g in all_circle_buffers[name]):return None
    if model=='M2' and rec.kind=='exhaustive' and int(rec.circle_id) not in DEV:return None
    inside=burn_window(d['roi_local'].geometry,name,win)
    pos_tree=burn_window(d['tree_positive'],name,win)
    neg_tree=burn_window(d['tree_negative'],name,win)
    if model=='M1':
        full=np.zeros_like(inside);pos_ex=np.zeros_like(inside)
    else:
        full=burn_window(d['dev_circles'].geometry,name,win)&inside
        pos_ex=burn_window(d['positive_local'].geometry,name,win)&full
    conflict=(pos_tree&neg_tree)| (pos_ex&neg_tree) | (full&~pos_ex&pos_tree)
    target=(pos_ex|(~full&pos_tree)).astype('float32')
    weight=(full|((pos_tree|neg_tree)&~full))&inside&~conflict
    img,valid=rgb_for_window(name,win)
    weight &= valid
    if int(weight.sum())<250:return None
    return dict(image=img,target=target,weight=weight.astype('float32'),
                site=d['site'],kind=rec.kind,mosaic=name)

def _pack_chips(path,items,signature):
    """Atomic, compressed uint8 cache of already-prepared chips."""
    if not items:return
    tmp=path.with_name(path.name+'.tmp.npz')
    try:
        np.savez_compressed(tmp,key=np.array(signature),
            rgb=np.stack([np.rint(np.clip(r['image'],0,1)*255).astype('uint8') for r in items]),
            target=np.stack([r['target'].astype('uint8') for r in items]),
            weight=np.stack([r['weight'].astype('uint8') for r in items]),
            site=np.array([r['site'] for r in items],dtype='U12'),
            kind=np.array([r['kind'] for r in items],dtype='U16'))
        tmp.replace(path)
    finally:
        tmp.unlink(missing_ok=True)

def _unpack_chips(path,signature):
    with np.load(path,allow_pickle=False) as z:
        assert str(z['key'].item())==signature
        rgb=z['rgb'];target=z['target'];weight=z['weight']
        sites=z['site'].astype(str);kinds=z['kind'].astype(str)
        assert rgb.ndim==4 and rgb.shape[1:]==(CROP,CROP,3)
        assert target.shape==weight.shape==rgb.shape[:3]
        return [dict(image=rgb[i].astype('float32')/255.,
                     target=target[i].astype('float32'),
                     weight=weight[i].astype('float32'),
                     site=str(sites[i]),kind=str(kinds[i]))
                for i in range(len(rgb))]

import shutil
from contextlib import contextmanager
LOCAL_STAGE=Path('/content/musanga_02f_stage')
LOCAL_STAGE.mkdir(parents=True,exist_ok=True)
MIN_FREE_AFTER_STAGE=2*1024**3
USE_LOCAL_STAGE=True

@contextmanager
def staged_mosaic(name):
    """Stage one raster on SSD if it fits; use one open raster handle per mosaic."""
    global _ACTIVE_RASTER,_ACTIVE_NAME
    source=Path(geo[name]['path'])
    assert source.exists(),source
    free=shutil.disk_usage('/content').free
    needed=source.stat().st_size
    local=LOCAL_STAGE/(name.replace('/','_'))
    can_stage=USE_LOCAL_STAGE and free>=needed+MIN_FREE_AFTER_STAGE
    selected_path=source
    if can_stage:
        print(f'STAGE {name}: {needed/1e9:.2f} GB from Drive to local SSD '
              f'(available {free/1e9:.1f} GB)',flush=True)
        t0=time.perf_counter()
        tmp=local.with_name(local.name+'.part')
        try:
            with source.open('rb') as inp,tmp.open('wb') as out:
                shutil.copyfileobj(inp,out,length=16*1024*1024)
            assert tmp.stat().st_size==needed
            tmp.replace(local)
            selected_path=local
            print(f'STAGED {name} in {time.perf_counter()-t0:.1f}s',flush=True)
        except Exception as exc:
            tmp.unlink(missing_ok=True)
            local.unlink(missing_ok=True)
            print(f'Local copy failed; using original Drive image: {exc}',flush=True)
    else:
        print(f'STAGE SKIPPED {name}: image {needed/1e9:.2f} GB, '
              f'free SSD {free/1e9:.2f} GB; using one persistent Drive handle',
              flush=True)
    assert _ACTIVE_RASTER is None,'Another mosaic is already open.'
    try:
        with rasterio.open(selected_path) as ds:
            assert ds.count>=3 and ds.transform==geo[name]['transform']
            assert ds.crs==geo[name]['crs']
            _ACTIVE_RASTER=ds
            _ACTIVE_NAME=name
            yield
    finally:
        _ACTIVE_RASTER=None
        _ACTIVE_NAME=None
        if selected_path==local:
            local.unlink(missing_ok=True)
            print(f'RELEASED local SSD for {name}',flush=True)

def make_training_cache(model):
    assert model=='M2'
    selected=[]
    for (site,kind),grp in candidate.groupby(['site','kind']):
        if kind!='partial':continue
        limit=MAX_PARTIAL_PER_MOSAIC*(3 if site=='MOS' else 1)
        subset=grp.sample(n=min(limit,len(grp)),random_state=SEED)
        selected.extend(subset.itertuples(index=False))
    selected_by_mosaic={}
    for rec in selected:
        selected_by_mosaic.setdefault(rec.mosaic,[]).append(rec)
    print('Selected M1 windows by mosaic:',flush=True)
    display(pd.DataFrame([
        dict(mosaic=name,site=geo[name]['site'],selected=len(rows))
        for name,rows in sorted(selected_by_mosaic.items())]))
    chunk_size=100
    parts_dir=CACHE_DIR/f'partial_{CACHE_KEY[:20]}'
    parts_dir.mkdir(parents=True,exist_ok=True)
    all_chips=[]
    t0=time.perf_counter()
    for mosaic_i,(name,records) in enumerate(sorted(selected_by_mosaic.items())):
        chunks=[]
        missing=0
        for offset in range(0,len(records),chunk_size):
            stop=min(offset+chunk_size,len(records))
            part=parts_dir/f'mosaic{mosaic_i:02d}_{offset:05d}_{stop:05d}.npz'
            marker=parts_dir/f'empty_mosaic{mosaic_i:02d}_{offset:05d}_{stop:05d}.txt'
            old=None
            if part.exists():
                try:
                    old=_unpack_chips(part,CACHE_KEY)
                except Exception as exc:
                    print('Discarding invalid chunk:',part.name,repr(exc),flush=True)
                    part.unlink(missing_ok=True)
            if old is None and marker.exists() and marker.read_text()==CACHE_KEY:
                old=[]
            if old is None:missing+=1
            chunks.append((offset,stop,part,marker,old))
        print(f'M1 {name}: {len(records)} candidates; '
              f'{len(chunks)-missing}/{len(chunks)} cached chunks',flush=True)
        if missing:
            with staged_mosaic(name):
                for offset,stop,part,marker,old in chunks:
                    if old is not None:continue
                    fresh=[]
                    for rec in records[offset:stop]:
                        chip=training_crop(rec,model)
                        if chip is not None:fresh.append(chip)
                    if fresh:
                        _pack_chips(part,fresh,CACHE_KEY)
                    else:
                        marker.write_text(CACHE_KEY)
                    print(f'M1 {name}: {stop}/{len(records)} attempted, '
                          f'{len(fresh)} usable in chunk, '
                          f'{(time.perf_counter()-t0)/60:.1f} min elapsed',flush=True)
        for offset,stop,part,marker,old in chunks:
            if old is not None:all_chips.extend(old)
            elif part.exists():all_chips.extend(_unpack_chips(part,CACHE_KEY))
            else:assert marker.exists() and marker.read_text()==CACHE_KEY
        print(f'M1 completed {name}; total usable chips {len(all_chips)}',flush=True)
    sites=pd.Series([x['site'] for x in all_chips]).value_counts().to_dict()
    print('M2 chips by site:',sites,'| total:',len(all_chips),
          '| elapsed minutes:',round((time.perf_counter()-t0)/60,2),flush=True)
    assert len(all_chips)>15
    return all_chips

class CropDataset(Dataset):
    def __init__(self,rows):self.rows=rows
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        q=self.rows[i];x=q['image'].copy();y=q['target'].copy();w=q['weight'].copy()
        k=int(torch.randint(0,4,(1,)).item())
        x=np.rot90(x,k);y=np.rot90(y,k);w=np.rot90(w,k)
        if torch.rand(1).item()<.5:x=x[:,::-1];y=y[:,::-1];w=w[:,::-1]
        return (torch.from_numpy(np.ascontiguousarray(x.transpose(2,0,1))),
                torch.from_numpy(np.ascontiguousarray(y)),torch.from_numpy(np.ascontiguousarray(w)))


# Cache only native, non-augmented crops; record source-file metadata and training rules.
import hashlib
CACHE_DIR=ROOT/'Outputs/V3/02D_M2_LOSO_no_MOS34'/'chip_cache'
CACHE_DIR.mkdir(parents=True,exist_ok=True)
def fingerprint():
    deps=[]
    paths=[REF/'01_candidate_windows.csv',
           ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json',
           A['roi'],A['exhaustive_annotations']['complete_roi_boundaries']]
    paths += [A['shared_annotations'][key] for key in sorted(A['shared_annotations'])]
    paths += [v['path'] for v in mosaics.values()]
    for path in paths:
        path=Path(path)
        more=[path.with_suffix(e) for e in ('.shp','.shx','.dbf','.prj','.cpg')
              if path.with_suffix(e).exists()] if path.suffix=='.shp' else [path]
        for p in more:
            deps.append([str(p),p.stat().st_size,p.stat().st_mtime_ns])
    spec=dict(version='M2_LOSO_noMOS34_current_ref_v1',dependencies=deps,
              seed=SEED,crop=CROP,holdout_ids=sorted(HOLDOUT),
              max_partial=MAX_PARTIAL_PER_MOSAIC,test_site=TEST_SITE,
              conflict_weight_zero=True)
    return hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest()
CACHE_KEY=fingerprint()
CACHE_FILE=CACHE_DIR/f'M1_{CACHE_KEY[:20]}.npz'
def get_m2_cache():
    if CACHE_FILE.exists():
        started=time.perf_counter()
        try:
            result=_unpack_chips(CACHE_FILE,CACHE_KEY)
            assert len(result)>15
            print('M1 FULL CACHE HIT:',len(result),'chips restored in',
                  round(time.perf_counter()-started,1),'seconds',flush=True)
            return result
        except Exception as e:
            print('Invalid full M1 cache, falling back to partial chunks:',repr(e),flush=True)
            CACHE_FILE.unlink(missing_ok=True)
    print('M1 cache miss: preparing source crops once.',flush=True)
    started=time.perf_counter()
    result=make_training_cache('M2')
    _pack_chips(CACHE_FILE,result,CACHE_KEY)
    print('M1 CACHE SAVED:',len(result),'chips in',
          round(time.perf_counter()-started,1),'seconds',flush=True)
    return result


## 05 — Four held-out sites: train M1, then evaluate independent partial tree labels

Each site model is fitted independently; MOS 1–2 and MOS 5–6 are held out together. AUC/AP use **only known positive/negative tree polygons** in the held-out site's candidate windows. Those windows may overlap, so treat scores as patch-sampled pixel discrimination, not an unbiased site-wide census. No exhaustive annotations are used for targets.


In [ ]:
import time
from contextlib import nullcontext
USE_AMP=(device=='cuda')
# float16 is fast on both T4 and L4; BCE is evaluated in fp32 for stability.
scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)
print('CUDA mixed precision:',USE_AMP,'| GPU:',torch.cuda.get_device_name(0),flush=True)

def fit_m2():
    ckpt=OUT/'M2_model.pt'
    torch.manual_seed(SEED)
    model=TinyUNet().to(device)
    if ckpt.exists():
        model.load_state_dict(torch.load(ckpt,map_location=device,weights_only=True))
        print('Reusing completed M1 partial-only checkpoint:',ckpt,flush=True)
        return model
    print('M2: checking persisted partial-only cache...',flush=True)
    prep=time.perf_counter()
    cache=get_m2_cache()
    print(f'M1: {len(cache)} usable training chips; prep {time.perf_counter()-prep:.1f}s',flush=True)
    loader=DataLoader(CropDataset(cache),batch_size=BATCH,shuffle=True,num_workers=0,
                      pin_memory=True)
    optim=torch.optim.AdamW(model.parameters(),lr=7e-4,weight_decay=1e-4)
    history=[]
    t0=time.perf_counter()
    for epoch in range(EPOCHS):
        model.train();losses=[];t_epoch=time.perf_counter()
        for step,(x,y,w) in enumerate(loader,1):
            x=x.to(device,non_blocking=True)
            y=y.to(device,non_blocking=True)
            w=w.to(device,non_blocking=True)
            optim.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=USE_AMP):
                logits=model(x)
            loss=masked_loss(logits.float(),y,w)
            scaler.scale(loss).backward()
            scaler.step(optim)
            scaler.update()
            losses.append(float(loss.detach()))
        elapsed=time.perf_counter()-t_epoch
        avg=float(np.mean(losses))
        history.append(dict(epoch=epoch+1,masked_BCE=avg,seconds=elapsed))
        pd.DataFrame(history).to_csv(OUT/'M1_loss.csv',index=False)
        print(f'M1 epoch {epoch+1:02d}/{EPOCHS}: BCE={avg:.4f}, '
              f'{elapsed:.1f}s, total {(time.perf_counter()-t0)/60:.1f} min',
              flush=True)
    torch.save(model.state_dict(),ckpt)
    print('M1 training complete; checkpoint:',ckpt,flush=True)
    del cache
    return model

print('Training function ready; execution is controlled by the LOSO loop below.',flush=True)


## 06 — Four M2 LOSO fits, testing on partial labels AND exhaustive circles

In [ ]:
from scipy.special import expit,logit
from scipy.optimize import minimize
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
WINDOWS=pd.read_csv(REF/'01_candidate_windows.csv')
VAL_WINDOWS=WINDOWS[WINDOWS.kind=='exhaustive'].copy()
VAL_WINDOWS=VAL_WINDOWS[VAL_WINDOWS.circle_id.isin(links.circle_id)].copy()
assert set(VAL_WINDOWS.circle_id.unique().astype(int))==set(links.circle_id.astype(int))
N_BINS=256
BINS=np.linspace(0,1,N_BINS+1)
BIN_CENTER=(BINS[:-1]+BINS[1:])/2

def run_all_circles(model,model_name):
    statfile=OUT/f'{model_name}_quick34_circle_raw.csv'
    histfile=OUT/f'{model_name}_quick34_histograms.npz'
    if statfile.exists() and histfile.exists():
        return pd.read_csv(statfile),dict(np.load(histfile))
    model.eval();rows=[];histograms={}
    for _,circle in split_df.iterrows():
        cid=int(circle.circle_id)
        name=circle.mosaic
        d=geo[name]
        windows=VAL_WINDOWS[(VAL_WINDOWS.circle_id==cid)&(VAL_WINDOWS.mosaic==name)]
        assert len(windows),f'No evaluation windows for circle {cid}'
        cg=circles[circles.original_row==cid].to_crs(d['crs'])
        counts=np.zeros(N_BINS,np.int64);true_counts=np.zeros(N_BINS,np.int64)
        n=0;sumy=0.;sump=0.;sum_sq=0.;tp=fp=fn=0
        rng=np.random.default_rng(SEED+cid)
        sample_truth=[];sample_probs=[]
        for rec in windows.itertuples(index=False):
            win=Window(int(rec.col),int(rec.row),int(rec.width),int(rec.height))
            mask=burn_window(cg.geometry,name,win)
            mask &= burn_window(d['roi_local'].geometry,name,win)
            rgb,valid=rgb_for_window(name,win)
            mask &= valid
            if not mask.any():continue
            reference=burn_window(d['positive_local'].geometry,name,win)
            x=torch.from_numpy(rgb.transpose(2,0,1).copy())[None].to(device)
            with torch.inference_mode():
                pred=torch.sigmoid(model(x))[0].cpu().numpy()
            y=reference[mask].astype('uint8');p=pred[mask].astype('float32')
            n+=len(y);sumy+=float(y.sum());sump+=float(p.sum())
            sum_sq+=float(np.square(p-y).sum())
            binary=p>=.5
            tp+=int(np.sum(binary&(y==1)));fp+=int(np.sum(binary&(y==0)))
            fn+=int(np.sum((~binary)&(y==1)))
            ix=np.minimum((p*N_BINS).astype(np.int32),N_BINS-1)
            counts+=np.bincount(ix,minlength=N_BINS).astype('int64')
            true_counts+=np.bincount(ix,weights=y,minlength=N_BINS).astype('int64')
            # only diagnostic AUC/AP uses pixel subsampling, with fixed per-circle cap
            take=rng.choice(len(y),min(100,len(y)),replace=False)
            sample_truth.extend(y[take].tolist());sample_probs.extend(p[take].tolist())
        assert n>0,f'No valid ROI/RGB reference pixels for circle {cid}'
        assert int(counts.sum())==n
        yt=np.asarray(sample_truth);pr=np.asarray(sample_probs)
        rows.append(dict(model=model_name,circle_id=cid,site=circle.site,split=circle.split,
                         valid_pixels=n,observed_fraction=100*sumy/n,
                         predicted_fraction=100*sump/n,
                         bias_pp=100*(sump-sumy)/n,brier=sum_sq/n,
                         f1_threshold_05=2*tp/max(1,2*tp+fp+fn),
                         auc_sample=roc_auc_score(yt,pr) if len(np.unique(yt))>1 else np.nan,
                         ap_sample=average_precision_score(yt,pr) if len(np.unique(yt))>1 else np.nan))
        histograms[str(cid)+'_count']=counts
        histograms[str(cid)+'_positive']=true_counts
        print(model_name,'circle',cid,'obs',round(100*sumy/n,1),
              'pred',round(100*sump/n,1),'valid pixels',n,flush=True)
    frame=pd.DataFrame(rows)
    assert frame.circle_id.is_unique
    assert set(frame.circle_id.astype(int))==set(split_df.circle_id.astype(int))
    assert len(frame)==len(split_df)
    frame.to_csv(statfile,index=False)
    np.savez_compressed(histfile,**histograms)
    return frame,histograms



In [ ]:
def evaluate_partial_heldout(model,test_site):
    model.eval()
    positives=np.zeros(256,dtype=np.float64)
    counts=np.zeros(256,dtype=np.float64)
    seen=0;masked_pixels=0
    for name in sorted(k for k,v in geo.items() if v['site']==test_site):
        records=candidate[(candidate.mosaic==name)&(candidate.kind=='partial')]
        # SSD staging also speeds up held-out inference.
        with staged_mosaic(name):
            for rec in records.itertuples(index=False):
                if rec.height<CROP or rec.width<CROP:continue
                rr=int(rec.row)+(int(rec.height)-CROP)//2
                cc=int(rec.col)+(int(rec.width)-CROP)//2
                win=Window(cc,rr,CROP,CROP)
                d=geo[name]
                y_pos=burn_window(d['tree_positive'],name,win)
                y_neg=burn_window(d['tree_negative'],name,win)
                label=(y_pos|y_neg)&~(y_pos&y_neg)
                label &= burn_window(d['roi_local'].geometry,name,win)
                rgb,valid=rgb_for_window(name,win)
                label &= valid
                if not label.any():continue
                with torch.inference_mode():
                    x=torch.from_numpy(rgb.transpose(2,0,1).copy())[None].to(device)
                    probs=torch.sigmoid(model(x))[0].detach().cpu().numpy()
                y=y_pos[label].astype('float64')
                p=probs[label]
                bix=np.minimum((p*256).astype('int32'),255)
                counts+=np.bincount(bix,minlength=256)
                positives+=np.bincount(bix,weights=y,minlength=256)
                seen+=1;masked_pixels+=int(label.sum())
        print(test_site,name,'evaluated windows:',seen,'pixels:',masked_pixels,flush=True)
    neg=counts-positives
    tp=float(positives.sum());tn=float(neg.sum())
    assert tp>0 and tn>0,(test_site,tp,tn)
    auc=float(np.dot(positives,np.cumsum(neg)-.5*neg)/(tp*tn))
    cumpos=np.cumsum(positives[::-1]);cumneg=np.cumsum(neg[::-1])
    ap=float(np.sum(positives[::-1]/tp*cumpos/np.maximum(cumpos+cumneg,1)))
    return dict(site=test_site,auc=auc,ap=ap,evaluated_windows=seen,
                known_pixels=int(masked_pixels),positive_pct=100*tp/(tp+tn))


from sklearn.metrics import r2_score,mean_absolute_error
import gc
SITES=['Baego','MOS','Yangambi','Yoko']
BASE_OUT=ROOT/'Outputs/V3/02D_M2_LOSO_no_MOS34'
M1_BASE=ROOT/'Outputs/V3/02A_partial_only_LOSO_no_MOS34'
M1_REF=ROOT/'Outputs/V3/02B_circle_LOSO_no_MOS34'/'02B_M1_all30_independent_circle_predictions.csv'
assert M1_REF.exists(),f'Run 02A and 02B first: {M1_REF}'
M1_REFERENCE=pd.read_csv(M1_REF)
assert M1_REFERENCE.circle_id.nunique()==30
VAL_WINDOWS=pd.read_csv(REF/'01_candidate_windows.csv')
VAL_WINDOWS=VAL_WINDOWS[(VAL_WINDOWS.kind=='exhaustive')&VAL_WINDOWS.circle_id.isin(links.circle_id)].copy()
# M2 sees training-site circles, but only withheld-site circles enter validation.
outputs=[]
partial_scores=[]
for TEST_SITE in SITES:
    OUT=BASE_OUT/TEST_SITE
    OUT.mkdir(parents=True,exist_ok=True)
    DEV=set(links.loc[links.site!=TEST_SITE,'circle_id'].astype(int))
    HOLDOUT=set(links.loc[links.site==TEST_SITE,'circle_id'].astype(int))
    split_df=links[links.site==TEST_SITE][['circle_id','site','mosaic','cover_pct']].copy()
    split_df['split']='independent_site_holdout'
    for name,d in geo.items():
        projected=circles.to_crs(d['crs'])
        d['dev_circles']=projected[projected.original_row.isin(DEV)]
        BLOCKED[name]=[g.buffer(15) for g in projected[projected.original_row.isin(HOLDOUT)].geometry]
    CACHE_KEY=fingerprint()
    CACHE_FILE=CACHE_DIR/f'M2_'+CACHE_KEY[:20]+'.npz'
    print('=== M2 LOSO site',TEST_SITE,'training circles',len(DEV),'test circles',len(HOLDOUT),'===',flush=True)
    model2=fit_m2()
    site_scores,_=run_all_circles(model2,'M2')
    partial_scores.append(evaluate_partial_heldout(model2,TEST_SITE))
    pd.DataFrame(partial_scores).to_csv(BASE_OUT/'02D_M2_LOSO_partial_AUC_AP.csv',index=False)
    assert set(site_scores.circle_id.astype(int))==HOLDOUT
    outputs.append(site_scores)
    del model2;torch.cuda.empty_cache();gc.collect()
M2=pd.concat(outputs,ignore_index=True)
assert len(M2)==30 and M2.circle_id.is_unique
M2.to_csv(BASE_OUT/'02D_M2_all30_independent_circle_predictions.csv',index=False)
paired=M1_REFERENCE[['circle_id','site','observed_fraction','predicted_fraction','bias_pp']].merge(
    M2[['circle_id','observed_fraction','predicted_fraction','bias_pp']],
    on='circle_id',suffixes=('_M1','_M2'),validate='one_to_one')
assert np.allclose(paired.observed_fraction_M1,paired.observed_fraction_M2,atol=0.02),(
    'M1 and M2 reference cover differ: check that both used the same refreshed NB01 inputs')
comparison=[]
for label in ['M1','M2']:
    y=paired['observed_fraction_'+label].to_numpy()
    p=paired['predicted_fraction_'+label].to_numpy()
    comparison.append(dict(model=label,n=len(y),R2=r2_score(y,p),
        MAE_pp=mean_absolute_error(y,p),bias_pp=float(np.mean(p-y))))
summary=pd.DataFrame(comparison)
display(summary.round(3))
print('M2 LOSO held-out partial-label pixel discrimination:')
display(pd.DataFrame(partial_scores).round(4))
M1_PARTIAL=M1_BASE/'02A_LOSO_partial_scores.csv'
if M1_PARTIAL.exists():
    qa=pd.read_csv(M1_PARTIAL)[['site','auc','ap']].merge(
        pd.DataFrame(partial_scores)[['site','auc','ap']],on='site',suffixes=('_M1','_M2'))
    display(qa.round(4))
    qa.to_csv(BASE_OUT/'02D_paired_partial_AUC_AP.csv',index=False)
display(paired.sort_values('bias_pp_M2',key=lambda x:x.abs(),ascending=False).round(2))
summary.to_csv(BASE_OUT/'02D_paired_summary.csv',index=False)
paired.to_csv(BASE_OUT/'02D_paired_circle_comparison.csv',index=False)
print('Completed LOSO M2 vs M1 comparison:',BASE_OUT)
